# summarize

> Generate section and full summaries from video xscripts via LLM

In [ ]:
#| default_exp summarize

## Design

One LLM call generates all section summaries + a full video summary.
Each section summary includes: summary text, keywords, and evidence quote.

`summaries.json` is **self-contained**: it embeds the relevant `meta.json` and `toc.json` fields so downstream consumers (CLI, learning-map generators, external tools) only need to open one file per video.

**summaries.json format:**
```json
{
  "video": {
    "id": "<VIDEO_ID>", "title": "...", "channel": "...",
    "url": "https://www.youtube.com/watch?v=<VIDEO_ID>",
    "duration": 7007, "upload_date": "20251101"
  },
  "sections": [
    {"path": "1", "title": "Introduction", "start": 0, "end": 137,
     "summary": "...", "keywords": [...],
     "evidence": {"text": "...", "at": 63}}
  ],
  "full": {"summary": "...", "keywords": [...], "evidence": {"text": "...", "at": 123}}
}
```

If `toc.json` is missing, `generate_toc` is called first internally. The LLM call returns `{full, sections: {path: {...}}}` and `_assemble_summaries` merges it with `meta` and `toc_sections` into the canonical shape above.

In [ ]:
#| export
import json
from pathlib import Path
from pydantic import BaseModel, Field
from yttoc.llm import generate_structured


In [ ]:
#| export
from yttoc.core import slice_segments, Segment, NormalizedSection, Meta

def _build_summary_prompt(segments: list[Segment], # Full xscript segments
                          sections: list[NormalizedSection], # List of NormalizedSection from toc.json
                          meta: Meta # Parsed Meta instance
                         ) -> str: # Prompt for LLM
    "Build prompt asking LLM to summarize each section and the full video."
    parts = []
    for sec in sections:
        sliced = slice_segments(segments, sec.start, sec.end)
        lines = []
        for s in sliced:
            mm = int(s.start // 60)
            ss = int(s.start % 60)
            lines.append(f"[{mm:02d}:{ss:02d}] {s.text}")
        parts.append(f"### Section {sec.path}: {sec.title}\n" + '\n'.join(lines))

    transcript = '\n\n'.join(parts)
    title = meta.title
    channel = meta.channel
    desc = meta.description

    return f"""You are a structural editor for YouTube video transcripts.

Video info:
- Title: {title}
- Channel: {channel}
- Description: {desc}

Transcript (organized by section):
{transcript}

Task:
For each section AND for the full video, provide:
- summary: 1-2 sentence English summary
- keywords: important terms (people, technical terms, proper nouns)
- evidence: a short quoted phrase from the transcript with its timestamp in seconds

Return a JSON object with:
- "full": {{summary, keywords, evidence: {{text, at}}}}
- "sections": {{"1": {{summary, keywords, evidence: {{text, at}}}}, "2": ...}}"""


In [ ]:
#| export
class Evidence(BaseModel):
    "A quoted phrase from the transcript with its timestamp."
    text: str = Field(description="Short quoted phrase from the transcript")
    at: int = Field(ge=0, description="Timestamp in seconds where the quote appears")

class SectionSummaryPayload(BaseModel):
    "Summary payload for one section or the full video."
    summary: str = Field(description="1-2 sentence English summary")
    keywords: list[str] = Field(description="Important terms (people, technical terms, proper nouns)")
    evidence: Evidence

class SummaryLLMResult(BaseModel):
    "Structured output from the summary generation LLM call."
    full: SectionSummaryPayload
    sections: dict[str, SectionSummaryPayload] = Field(
        description="Per-section summaries keyed by section path ('1', '2', ...)")

class VideoBlock(BaseModel):
    "Video header subset persisted inside summaries.json."
    id: str = Field(description="YouTube video ID")
    title: str = Field(description="Video title")
    channel: str = Field(description="Channel name")
    url: str = Field(description="Canonical YouTube URL")
    duration: int = Field(ge=0, description="Duration in seconds")
    upload_date: str = Field(description="Upload date in YYYYMMDD format")

class AssembledSection(NormalizedSection):
    "TOC section with LLM-generated summary payload."
    summary: str = Field(description="1-2 sentence summary")
    keywords: list[str] = Field(description="Important terms")
    evidence: Evidence = Field(description="Quoted phrase + timestamp")

class AssembledSummaries(BaseModel):
    "On-disk shape of summaries.json."
    video: VideoBlock
    sections: list[AssembledSection]
    full: SectionSummaryPayload

def _call_summary_llm(prompt: str) -> dict:
    "Call OpenAI gpt-5.4 with Pydantic-generated schema, return {full, sections} dict."
    return generate_structured(
        prompt, SummaryLLMResult, schema_name='generate_summaries').model_dump()

## Tests

In [ ]:
from yttoc.core import Segment
# Test 1: slice_segments returns segments within time range
segs = [
    Segment(start=0, end=5, text='a'),
    Segment(start=5, end=10, text='b'),
    Segment(start=10, end=15, text='c'),
    Segment(start=15, end=20, text='d'),
]
sliced = slice_segments(segs, start=5, end=15)
assert len(sliced) == 2
assert sliced[0].text == 'b'
assert sliced[1].text == 'c'
print('ok')

In [ ]:
# Test 2: slice_segments with no matching segments returns empty
sliced = slice_segments(segs, start=100, end=200)
assert sliced == []
print('ok')

In [ ]:
from yttoc.core import Segment, Meta
from datetime import datetime, timezone
# Test 3: _build_summary_prompt includes section titles and transcript
segments = [
    Segment(start=0, end=5, text='hello world'),
    Segment(start=5, end=10, text='second part'),
]
from yttoc.core import NormalizedSection
sections = [
    NormalizedSection(path='1', title='Intro', start=0, end=5),
    NormalizedSection(path='2', title='Main', start=5, end=10),
]
meta = Meta(
    id='T', title='Test Video', channel='Ch',
    duration=600, upload_date='20260101',
    webpage_url='https://youtube.com/watch?v=T',
    description='', captions={'en': 'auto'},
    last_used_at=datetime(2026, 1, 1, tzinfo=timezone.utc))
prompt = _build_summary_prompt(segments, sections, meta)
assert 'Test Video' in prompt
assert 'Intro' in prompt
assert 'Main' in prompt
assert 'hello world' in prompt
assert 'second part' in prompt
print('ok')


In [ ]:
# Test: VideoBlock, AssembledSection, AssembledSummaries validate required fields and inheritance
from yttoc.summarize import VideoBlock, AssembledSection, AssembledSummaries
from yttoc.core import NormalizedSection
from pydantic import ValidationError

# VideoBlock valid
vb = VideoBlock(id='x', title='t', channel='c', url='u', duration=60, upload_date='20260101')
assert vb.id == 'x' and vb.duration == 60

# VideoBlock rejects missing required field
try:
    VideoBlock(id='x', title='t', channel='c', url='u', duration=60)
except ValidationError:
    pass
else:
    assert False, 'expected ValidationError for missing upload_date'

# AssembledSection inherits NormalizedSection
assert issubclass(AssembledSection, NormalizedSection)
as_ = AssembledSection(path='1', title='Intro', start=0, end=300,
                       summary='s', keywords=['k'],
                       evidence={'text': 'e', 'at': 0})
assert isinstance(as_, NormalizedSection)
assert as_.path == '1' and as_.summary == 's' and as_.evidence.at == 0

# AssembledSection rejects missing summary
try:
    AssembledSection(path='1', title='t', start=0, end=10,
                     keywords=[], evidence={'text': '', 'at': 0})
except ValidationError:
    pass
else:
    assert False, 'expected ValidationError for missing summary'

# AssembledSummaries envelope validates via model_validate_json
doc = '{"video": {"id": "X", "title": "T", "channel": "C", "url": "u", "duration": 60, "upload_date": "20260101"},'
doc += '"sections": [{"path": "1", "title": "I", "start": 0, "end": 30, "summary": "s", "keywords": ["k"], "evidence": {"text": "e", "at": 0}}],'
doc += '"full": {"summary": "f", "keywords": ["fk"], "evidence": {"text": "fe", "at": 0}}}'
doc_model = AssembledSummaries.model_validate_json(doc)
assert doc_model.video.id == 'X'
assert len(doc_model.sections) == 1
assert doc_model.sections[0].title == 'I'

# AssembledSummaries rejects missing top-level envelope key
try:
    AssembledSummaries.model_validate_json('{"sections": [], "full": {"summary": "s", "keywords": [], "evidence": {"text": "", "at": 0}}}')
except ValidationError:
    pass
else:
    assert False, 'expected ValidationError for missing video key'

print('ok')


## CLI

In [ ]:
#| export
from fastcore.script import call_parse
from yttoc.core import fmt_duration, format_header, format_toc_line, NormalizedSection, Meta
from yttoc.cache import (resolve_root, meta_path, summaries_path,
                         first_srt_path, load_meta, read_model, touch_meta)
from yttoc.xscript import parse_xscript
from yttoc.toc import generate_toc

def _assemble_summaries(meta: Meta, # Parsed Meta instance
                        toc_sections: list[NormalizedSection], # List of NormalizedSection from toc.json
                        llm_result: dict # {full, sections: {path: {...}}}
                       ) -> AssembledSummaries: # Parsed AssembledSummaries instance
    "Merge meta + toc + LLM output into the canonical summaries.json shape. Raise if LLM omitted any section."
    missing = [sec.path for sec in toc_sections if sec.path not in llm_result['sections']]
    if missing:
        raise ValueError(f"LLM omitted summaries for sections: {missing}")
    return AssembledSummaries(
        video=VideoBlock(
            id=meta.id,
            title=meta.title,
            channel=meta.channel,
            url=meta.webpage_url,
            duration=meta.duration,
            upload_date=meta.upload_date,
        ),
        sections=[
            AssembledSection(**sec.model_dump(), **llm_result['sections'][sec.path])
            for sec in toc_sections
        ],
        full=llm_result['full'],
    )

def generate_summaries(video_id: str, # Exact video_id
                       root: Path = None, # Root cache directory
                       refresh: bool = False, # Delete cached summaries and regenerate
                      ) -> AssembledSummaries: # Parsed AssembledSummaries instance
    "Generate summaries.json for a cached video. Returns parsed AssembledSummaries."
    root = resolve_root(root)
    meta_p = meta_path(video_id, root)
    sum_p = summaries_path(video_id, root)
    if not meta_p.exists():
        raise SystemExit(f"Not cached: {video_id}")
    try:
        srt_path = first_srt_path(video_id, root)
    except FileNotFoundError:
        raise SystemExit(f"Not cached: {video_id}")

    if refresh and sum_p.exists():
        sum_p.unlink()

    if sum_p.exists():
        return read_model(sum_p, AssembledSummaries)

    toc_sections = generate_toc(video_id, root)
    meta = load_meta(video_id, root)
    segments = parse_xscript(srt_path)
    prompt = _build_summary_prompt(segments, toc_sections, meta)
    llm_result = _call_summary_llm(prompt)
    result = _assemble_summaries(meta, toc_sections, llm_result)

    sum_p.write_text(result.model_dump_json(indent=2), encoding='utf-8')
    touch_meta(video_id, root)
    return result

def _format_section_summary(s: AssembledSection, # Assembled section with summary payload
                            url: str, # Canonical video URL ('' when absent)
                           ) -> list[str]: # Four-line block: TOC heading, summary, keywords, evidence
    "Format one section as a TOC-heading block."
    return [
        f"## {format_toc_line(s, url)}",
        s.summary,
        f"**Keywords:** {', '.join(s.keywords)}",
        f"**Evidence:** \"{s.evidence.text}\" [{fmt_duration(s.evidence.at)}]",
    ]

def _render_summaries(sums: AssembledSummaries, # Parsed summaries.json
                     section: str = '', # Section path; '' for all sections + full summary
                    ) -> str: # Rendered summaries output
    "Render summary output for yttoc_sum. Raise ValueError if section missing."
    url = sums.video.url or ''
    lines = [format_header(sums.video), '']
    if section:
        s = next((sec for sec in sums.sections if sec.path == section), None)
        if s is None:
            raise ValueError(f"Section {section} not found")
        lines.extend(_format_section_summary(s, url))
    else:
        for s in sums.sections:
            lines.extend(_format_section_summary(s, url))
            lines.append('')
        lines.append("## Full Summary")
        lines.append(sums.full.summary)
        lines.append(f"**Keywords:** {', '.join(sums.full.keywords)}")
        lines.append(f"**Evidence:** \"{sums.full.evidence.text}\" [{fmt_duration(sums.full.evidence.at)}]")
        if url: lines.append(url)
    return '\n'.join(lines)

@call_parse
def yttoc_sum(video_id: str, # Exact video_id
              section: str = '', # Section path (e.g. "3"); empty for all
              root: str = None, # Root cache directory
              refresh: bool = False, # Regenerate summaries
             ):
    "Display summaries for a cached video."
    root = resolve_root(root)
    sums = generate_summaries(video_id, root, refresh=refresh)
    try:
        print(_render_summaries(sums, section))
    except ValueError as e:
        raise SystemExit(str(e))

In [ ]:
# Test 4: generate_summaries returns cached summaries.json without LLM call
from tempfile import TemporaryDirectory
import io, contextlib

def _make_test_summaries(video_id: str, url: str = ''):
    "Build a self-contained summaries.json fixture with the given id/url."
    return {
        'video': {
            'id': video_id, 'title': 'T', 'channel': 'C',
            'url': url, 'duration': 600, 'upload_date': '20260101',
        },
        'sections': [
            {'path': '1', 'title': 'Intro', 'start': 0, 'end': 300,
             'summary': 'Intro section.', 'keywords': ['intro'],
             'evidence': {'text': 'hi', 'at': 0}},
            {'path': '2', 'title': 'Main', 'start': 300, 'end': 600,
             'summary': 'Main section.', 'keywords': ['main'],
             'evidence': {'text': 'bye', 'at': 300}},
        ],
        'full': {'summary': 'Full video about testing.', 'keywords': ['test'],
                 'evidence': {'text': 'hello', 'at': 0}},
    }

with TemporaryDirectory() as d:
    root = Path(d)
    v = root / 'VID1'; v.mkdir()
    (v / 'captions.en.srt').write_text('1\n00:00:00,000 --> 00:00:01,000\nhi\n')
    (v / 'meta.json').write_text(json.dumps({
        'id': 'VID1', 'title': 'T', 'channel': 'C', 'duration': 600,
        'upload_date': '20260101', 'webpage_url': 'https://youtube.com/watch?v=VID1',
        'description': '', 'captions': {'en': 'auto'},
        'last_used_at': '2000-01-01T00:00:00+00:00',
    }))
    (v / 'summaries.json').write_text(json.dumps(_make_test_summaries('VID1')))

    result = generate_summaries('VID1', root)
    assert result.full.summary == 'Full video about testing.'
    assert result.video.id == 'VID1'
    assert isinstance(result.sections, list)
    assert len(result.sections) == 2
    assert {s.path for s in result.sections} == {'1', '2'}
print('ok')


In [ ]:
# Test 5: yttoc_sum reads summaries.json only (no toc.json), embedded URL flows through
with TemporaryDirectory() as d:
    root = Path(d)
    v = root / 'VID2'; v.mkdir()
    (v / 'captions.en.srt').write_text('1\n00:00:00,000 --> 00:00:01,000\nhi\n')
    (v / 'meta.json').write_text(json.dumps({
        'id': 'VID2', 'title': 'Test Video', 'channel': 'Ch', 'duration': 600,
        'upload_date': '20260101', 'webpage_url': 'https://youtube.com/watch?v=VID2',
        'description': '', 'captions': {'en': 'auto'},
        'last_used_at': '2000-01-01T00:00:00+00:00',
    }))
    fixture = _make_test_summaries('VID2', url='https://youtube.com/watch?v=VID2')
    fixture['video']['title'] = 'Test Video'
    fixture['video']['channel'] = 'Ch'
    (v / 'summaries.json').write_text(json.dumps(fixture))
    # Intentionally NO toc.json — yttoc_sum must not depend on it.

    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        yttoc_sum('VID2', root=str(root))
    out = buf.getvalue()

    assert '# Test Video' in out
    assert '## 1. Intro 0:00-5:00 (5:00) https://youtube.com/watch?v=VID2&t=0' in out
    assert '## 2. Main 5:00-10:00 (5:00) https://youtube.com/watch?v=VID2&t=300' in out
    assert 'Intro section.' in out
    assert '**Keywords:** intro' in out
    assert '## Full Summary' in out
    assert 'Full video about testing.' in out
    assert 'https://youtube.com/watch?v=VID2' in out  # full-summary URL line
print('ok')

In [ ]:
# Test 6: _render_summaries with single section
from yttoc.summarize import _render_summaries

sums_dict = _make_test_summaries('VID3')
sums = AssembledSummaries.model_validate(sums_dict)

out = _render_summaries(sums, '2')
assert '## 2. Main 5:00-10:00 (5:00)' in out
assert '&t=' not in out  # no URL in test fixture -> no deep link
assert 'Main section.' in out
assert 'Intro section.' not in out
assert '## Full Summary' not in out
print('ok')


In [ ]:
# Test: _render_summaries (all sections + full) returns expected block
from yttoc.summarize import _render_summaries

sums_dict = _make_test_summaries('VID9', url='https://youtube.com/watch?v=VID9')
sums_dict['video']['title'] = 'Test Video'
sums_dict['video']['channel'] = 'Ch'
sums = AssembledSummaries.model_validate(sums_dict)

out = _render_summaries(sums, '')
assert '# Test Video' in out
assert '## 1. Intro' in out
assert '## 2. Main' in out
assert '## Full Summary' in out
assert 'Full video about testing.' in out
assert out.endswith('https://youtube.com/watch?v=VID9')  # url footer
print('ok')

In [ ]:
# Test: _render_summaries(section='2') returns one section; missing section raises ValueError
from yttoc.summarize import _render_summaries

sums = AssembledSummaries.model_validate(_make_test_summaries('VIDX'))

out = _render_summaries(sums, '2')
assert '## 2. Main' in out
assert '## 1. Intro' not in out
assert '## Full Summary' not in out

try:
    _render_summaries(sums, '99')
except ValueError as e:
    assert 'Section 99 not found' in str(e)
else:
    raise AssertionError('expected ValueError for missing section')
print('ok')

In [ ]:
# Test 8: _assemble_summaries raises if LLM omits any toc section (no silent corruption)
from yttoc.core import NormalizedSection
toc = [
    NormalizedSection(path='1', title='A', start=0, end=100),
    NormalizedSection(path='2', title='B', start=100, end=200),
]
llm_partial = {
    'full': {'summary': 'f', 'keywords': [], 'evidence': {'text': '', 'at': 0}},
    'sections': {
        '1': {'summary': 's1', 'keywords': [], 'evidence': {'text': '', 'at': 0}},
        # '2' is missing
    },
}
try:
    _assemble_summaries({'id': 'X', 'title': 't'}, toc, llm_partial)
    assert False, 'should have raised'
except ValueError as e:
    assert "'2'" in str(e)
print('ok')


## get_summaries

In [ ]:
#| export
def _get_summaries_strict(video_id: str, # Exact video_id
                          root: Path = None # Root cache directory (default: ~/.cache/yttoc)
                         ) -> AssembledSummaries: # Parsed AssembledSummaries
    "Return validated summaries.json or raise on missing/invalid data."
    root = resolve_root(root)
    sum_p = summaries_path(video_id, root)
    if not sum_p.exists():
        raise FileNotFoundError(f'summaries.json not found for {video_id}')
    return read_model(sum_p, AssembledSummaries)

def get_summaries(video_id: str, # Exact video_id
                  root: Path = None # Root cache directory (default: ~/.cache/yttoc)
                 ) -> AssembledSummaries | dict: # Parsed AssembledSummaries or {"error": "..."}
    "Compatibility wrapper around _get_summaries_strict for tool/CLI consumers expecting {'error': ...}."
    from pydantic import ValidationError
    try:
        return _get_summaries_strict(video_id, root)
    except FileNotFoundError as e:
        return {'error': str(e)}
    except ValidationError as e:
        return {'error': f'Invalid summaries.json for {video_id}: {e}'}

In [ ]:
# Test 9: _get_summaries_strict returns AssembledSummaries
with TemporaryDirectory() as d:
    root = Path(d)
    v = root / 'VID_GS'; v.mkdir()
    fixture = {
        'video': {'id': 'VID_GS', 'title': 'T', 'channel': 'C',
                  'url': '', 'duration': 600, 'upload_date': '20260101'},
        'sections': [
            {'path': '1', 'title': 'Intro', 'start': 0, 'end': 300,
             'summary': 's', 'keywords': ['k'],
             'evidence': {'text': 'e', 'at': 10}}
        ],
        'full': {'summary': 'full', 'keywords': ['fk'],
                 'evidence': {'text': 'fe', 'at': 0}},
    }
    (v / 'summaries.json').write_text(json.dumps(fixture))

    result = _get_summaries_strict('VID_GS', root)
    from yttoc.summarize import AssembledSummaries
    assert isinstance(result, AssembledSummaries)
    assert result.video.id == 'VID_GS'
    assert result.sections[0].title == 'Intro'
    assert result.full.summary == 'full'
print('ok')


In [ ]:
# Test: get_summaries wrapper preserves {'error': ...} contract when missing
with TemporaryDirectory() as d:
    result = get_summaries('NONEXIST', Path(d))
    assert 'error' in result
print('ok')

In [ ]:
# Test 10: _get_summaries_strict raises FileNotFoundError when missing
with TemporaryDirectory() as d:
    try:
        _get_summaries_strict('NONEXIST', Path(d))
    except FileNotFoundError as e:
        assert 'NONEXIST' in str(e)
    else:
        assert False, 'expected FileNotFoundError'
print('ok')

In [ ]:
# Test: _get_summaries_strict rejects a corrupted summaries.json (missing evidence field)
from tempfile import TemporaryDirectory
from pydantic import ValidationError

with TemporaryDirectory() as d:
    root = Path(d)
    v = root / 'BAD_SUM'; v.mkdir()
    # Bad shape: section is missing `evidence`
    (v / 'summaries.json').write_text(json.dumps({
        'video': {'id': 'BAD_SUM', 'title': 'T', 'channel': 'C',
                  'url': '', 'duration': 60, 'upload_date': '20260101'},
        'sections': [
            {'path': '1', 'title': 'I', 'start': 0, 'end': 10,
             'summary': 's', 'keywords': ['k']}  # no 'evidence'
        ],
        'full': {'summary': 'f', 'keywords': ['fk'], 'evidence': {'text': 'fe', 'at': 0}},
    }))

    try:
        _get_summaries_strict('BAD_SUM', root)
    except ValidationError as e:
        assert 'evidence' in str(e)
    else:
        assert False, 'expected ValidationError'

    result = get_summaries('BAD_SUM', root)
    assert isinstance(result, dict)
    assert 'error' in result
    assert 'BAD_SUM' in result['error']
print('ok')

In [ ]:
# Test: AssembledSummaries round-trip preserves fields through model_dump_json -> model_validate_json
from yttoc.summarize import AssembledSummaries, VideoBlock, AssembledSection

original = AssembledSummaries(
    video=VideoBlock(id='R', title='T', channel='C', url='u',
                     duration=60, upload_date='20260101'),
    sections=[
        AssembledSection(path='1', title='I', start=0, end=30,
                         summary='s', keywords=['k'],
                         evidence={'text': 'e', 'at': 5}),
    ],
    full={'summary': 'f', 'keywords': ['fk'], 'evidence': {'text': 'fe', 'at': 0}},
)
serialized = original.model_dump_json(indent=2)
reparsed = AssembledSummaries.model_validate_json(serialized)
assert reparsed == original, 'round-trip mismatch'
assert reparsed.sections[0].evidence.at == 5
print('ok')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()